# Week 1 — BANKING77 / RoBERTa pilots (Kunj)
**Kaggle settings (right panel):** Accelerator = **GPU T4 x2**, Internet = **On**.
We use only ONE T4 (proposal setup). At the end, download `week1_results.zip` from the Output panel and commit it from your laptop.

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # use exactly one T4, matching the proposal
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Clone the repo. If it is PRIVATE: Add-ons -> Secrets -> add GITHUB_TOKEN, and attach it to this notebook.
import os, subprocess
url = 'https://github.com/devjp99/lora-on-a-budget.git'
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret('GITHUB_TOKEN')
    url = f'https://{tok}@github.com/devjp99/lora-on-a-budget.git'
    print('using GITHUB_TOKEN secret')
except Exception:
    print('no token secret found; cloning as public repo')
subprocess.run(['git', 'clone', url], check=True)
%cd lora-on-a-budget
!git checkout kunj/week1-banking77

In [ ]:
!pip uninstall -y -q torchao   # Kaggle ships torchao 0.10, which peft rejects; we don't use it
!pip install -q -r requirements.txt
!pip install -q -e .
os.environ['PYTHONPATH'] = 'src'

## 1. Build manifests (should match the committed ones exactly)

In [ ]:
!python scripts/make_banking77_manifest.py
!git status --short manifests/   # empty output = identical to what we committed

## 2. Tests with the REAL pretrained roberta-base

In [ ]:
!LORA_BUDGET_ONLINE=1 python -m pytest -q

## 3. Parameter audit vs proposal Table 1

In [ ]:
!python scripts/param_audit_roberta.py 2>&1 | tail -1

## 4. 20-step pilots for the three E1 setups

In [ ]:
for c in ["full_ft", "head_only", "lora_r8"]:
    !python scripts/pilot_banking77.py --config configs/banking77/{c}.yaml

In [ ]:
import pandas as pd
pd.read_csv('results/run_ledger.csv')[['method','status','final_loss','peak_alloc_mb_steady','mean_step_s','checkpoint_mb','projected_run_gpu_h']]

## 5. Save results for download (checkpoints stay out of git)

In [ ]:
import shutil, os
out_dir = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
shutil.make_archive(os.path.join(out_dir, 'week1_results'), 'zip', '.', 'results')
print('saved', os.path.join(out_dir, 'week1_results.zip'))
try:
    from google.colab import files; files.download('week1_results.zip')  # Colab only
except Exception:
    print('On Kaggle: download week1_results.zip from the Output panel on the right')